# 歌词采集

In [44]:
import requests
import re
import json
import os
import time
import pandas as pd
from collections import defaultdict

import jieba
import jieba.posseg as pseg
from collections import Counter
from openai import OpenAI

In [31]:
jieba.load_userdict('data/mayday_dict.txt')

In [4]:
import sys
sys.path.append('..')

# 原始数据清洗

In [5]:
df_raw = pd.read_excel('data/mayday_songs.xlsx')
df_raw

,album_order,album_id,album_name,release_date,song_order,song_id,song_name,album_fixed
0,1,38315,第一张创作专辑,1999-07-07,1,386925,疯狂世界,第一张创作专辑
1,1,38315,第一张创作专辑,1999-07-07,2,386927,拥抱,第一张创作专辑
2,1,38315,第一张创作专辑,1999-07-07,3,386929,透露,第一张创作专辑
3,1,38315,第一张创作专辑,1999-07-07,4,386930,生活,第一张创作专辑
4,1,38315,第一张创作专辑,1999-07-07,5,386931,爱情的模样,第一张创作专辑
...,...,...,...,...,...,...,...,...
155,10,38247,知足 最真杰作选,2005-08-26,26,386030,叫我第一名,知足 最真杰作选
156,10,38247,知足 最真杰作选,2005-08-26,27,386035,永远的永远,知足 最真杰作选
157,10,38247,知足 最真杰作选,2005-08-26,28,386036,借问众神明,知足 最真杰作选
158,10,38247,知足 最真杰作选,2005-08-26,29,386038,未来 (Sailing With Me),知足 最真杰作选


In [52]:
df_raw['song_name'].value_counts()

song_name
疯狂世界                2
有些事现在不做 一辈子都不会做了    2
好不好                 2
星空                  2
洗衣机                 2
                   ..
生命有一种绝对             1
在这一秒                1
我们 (时时刻刻)           1
时光机                 1
咸鱼                  1
Name: count, Length: 130, dtype: int64

In [24]:
df_unique = df_raw.drop_duplicates(subset='song_name', keep='first').copy()
df_unique['song_name'] = df_unique['song_name'].astype(str)
df_unique

,album_order,album_id,album_name,release_date,song_order,song_id,song_name,album_fixed
0,1,38315,第一张创作专辑,1999-07-07,1,386925,疯狂世界,第一张创作专辑
1,1,38315,第一张创作专辑,1999-07-07,2,386927,拥抱,第一张创作专辑
2,1,38315,第一张创作专辑,1999-07-07,3,386929,透露,第一张创作专辑
3,1,38315,第一张创作专辑,1999-07-07,4,386930,生活,第一张创作专辑
4,1,38315,第一张创作专辑,1999-07-07,5,386931,爱情的模样,第一张创作专辑
...,...,...,...,...,...,...,...,...
149,10,38247,知足 最真杰作选,2005-08-26,20,386010,麦来乱,知足 最真杰作选
153,10,38247,知足 最真杰作选,2005-08-26,24,386025,OK 啦,知足 最真杰作选
154,10,38247,知足 最真杰作选,2005-08-26,25,386028,垃圾车 (朋友版),知足 最真杰作选
158,10,38247,知足 最真杰作选,2005-08-26,29,386038,未来 (Sailing With Me),知足 最真杰作选


# 歌词采集

In [41]:
# --- 核心清洗函数 ---
def clean_and_format_lyrics(raw_text):
    if not raw_text or "未能获取" in raw_text:
        return ""
    
    # 1. 预处理：处理特殊空格 U+00A0
    text = raw_text.replace('\u00a0', ' ')
    
    # 2. 定义黑名单关键词
    exclude_keywords = [
        '作词', '作曲', '编曲', '：', ':', '演奏', '吉他', '贝斯', 
        '鼓', '编写', '演唱', '合唱', '制作', '录音', '混音', 'ISRC', 
        '编码', '版权', '提供', '发行', 'OP', 'SP'
    ]
    
    # 3. 专门针对 ISRC 这种特征码的正则表达式
    # 匹配规律：大写字母开头，中间有多个连字符和数字，例如 TW-K23-08-016-11
    isrc_pattern = r'[A-Z]{2}-[A-Z0-9]{3}-\d{2}-\d{5}'
    
    pure_lyrics_list = []
    lines = text.split('\n')
    
    for line in lines:
        # 提取时间戳后面的内容
        match = re.search(r'\[.*\]\s*(.*)', line)
        if match:
            content = match.group(1).strip()
            
            # --- 过滤逻辑开始 ---
            # A. 检查是否为空
            if not content:
                continue
            
            # B. 检查是否包含黑名单关键词
            if any(k in content.upper() for k in exclude_keywords): # 转大写匹配，防止漏掉 isrc
                continue
                
            # C. 检查是否匹配 ISRC 正则特征
            if re.search(isrc_pattern, content):
                continue
            
            # --- 过滤逻辑结束 ---
            
            # 内部空格换逗号，去除多余空白
            clean_content = re.sub(r'\s+', ' ', content).replace(" ", "，")
            pure_lyrics_list.append(clean_content)
    
    # 用句号连接
    if not pure_lyrics_list: return ""
    return "。".join(pure_lyrics_list) + "。"

# --- 制作信息提取 ---
def get_credit_info(raw_text):
    """从原始文本中提取 作词/作曲/编曲"""
    info = {"作词": "", "作曲": "", "编曲": ""}
    # 兼容带时间戳和不带时间戳的情况
    for key in info.keys():
        pattern = rf"{key}\s*[:：]\s*([^\]\n]+)"
        match = re.search(pattern, raw_text)
        if match:
            # 清理掉可能残余的括号或空格
            info[key] = match.group(1).strip()
    return info

# --- API 请求函数 ---
def get_lyrics_by_api(song_id):
    api_url = f"https://music.163.com/api/song/lyric?id={song_id}&lv=1&kv=1&tv=-1"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    try:
        response = requests.get(api_url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data.get('lrc', {}).get('lyric', "")
    except Exception as e:
        print(f"ID {song_id} 获取失败: {e}")
        return ""

# --- 主逻辑封装 ---
def process_single_song(song_id, song_name):
    """处理单首歌曲：下载 -> 解析 -> 组装字典"""
    raw_lyric = get_lyrics_by_api(song_id)
    
    if not raw_lyric:
        return None
    
    # 提取制作人信息
    credits = get_credit_info(raw_lyric)
    # 提取并清洗歌词
    formatted_lyric = clean_and_format_lyrics(raw_lyric)
    
    # 组装结果
    return {
        "歌名": song_name,
        "song_id": song_id,
        "作词": credits["作词"],
        "作曲": credits["作曲"],
        "编曲": credits["编曲"],
        "歌词": formatted_lyric
    }

# --- 文件保存（增量更新） ---
def save_to_json_list(file_path, song_data):
    """以列表形式保存所有歌曲，避免字典 key 覆盖的问题"""
    data_list = []
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data_list = json.load(f)
                if not isinstance(data_list, list): data_list = []
            except:
                data_list = []
    
    data_list.append(song_data)
    
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_list, f, ensure_ascii=False, indent=4)

In [ ]:
# 测试
id_1 = "385563"
song_name_1 = 'test_1'
process_single_song(id_1, song_name_1)

{'歌名': 'test_1',
 'song_id': '385563',
 '作词': '五月天 阿信',
 '作曲': '五月天 阿信',
 '编曲': '五月天',
 '歌词': '舔一下，饥渴的味蕾。跳一下，期盼的脚尖。恨一种，无止尽的和谐。等一个，复活的瞬间。喝一滴，甜美的眼泪。唱一首，透明摇滚乐。创一个，颠峰的疯癫。这一生，太多的妥协。这一刻，彻底的打碎。这一次，丢掉名字性别。快张开你的嘴，OA，OA。再不管你是谁，OA，OA。人生都太短暂。别想，别怕，别後退。现在，就是，永远。出生的那一年，OA，OA。转眼就这一天，OA，OA。人生都太短暂。去疯，去爱，去浪费。和我，再唱，OA，OAOA。舔一下，饥渴的味蕾。跳一下，期盼的脚尖。恨一种，无止尽的和谐。等一个，复活的瞬间。喝一滴，甜美的眼泪。唱一首，透明摇滚乐。创一个，颠峰的疯癫。这一生，太多的妥协。这一刻，彻底的打碎。这一次，丢掉名字性别。快张开你的嘴，OA，OA。再不管你是谁，OA，OA。人生都太短暂。别想，别怕，别後退。现在，就是，永远。出生的那一年，OA，OA。转眼就这一天，OA，OA。人生都太短暂。去疯，去爱，去浪费。和我，再唱，OA，OAOA。这一生，太多的妥协。这一刻，彻底的打碎。这一次，丢掉名字性别。快张开你的嘴，OA，OA。再不管你是谁，OA，OA。人生都太短暂。别想，别怕，别後退。现在，就是，永远。出生的那一年，OA，OA。转眼就这一天，OA，OA。人生都太短暂。去疯，去爱，去浪费。和我，再唱，OA，OAOA。O，A，O，A。O，A，O，A。O，A，O，A。O，A，O，A。O，A，O，A。O，A，O，A。O，A，O，A。O，A，O，A。O，A，O，A。'}

In [43]:
file_path = 'output/mayday_lyric.json'
# 遍历df_df_unique每一行
for index, row in df_unique.iterrows():
    song_id = str(row['song_id'])
    song_name = row['song_name']
    print(song_name)
    single_res = process_single_song(song_id, song_name)
    save_to_json_list(file_path, single_res)
    time.sleep(2)

疯狂世界
拥抱
透露
生活
爱情的模样
嘿！我要走了
轧车
志明与春娇
HoSee
黑白讲
I Love You 无望
风若吹
为什么 (今日的爱情)
终结孤单
明白
心中无别人
有你的将来
憨人
叫我第一名
雨眠
罗密欧与茱丽叶
温柔
爱情万岁
反而
一颗苹果
能不能不要说
好不好
相信
Ok啦
借问众神明
永远的永远
彩虹
啾啾啾
纯真
候鸟
人生海海
轻功
恒星的恒心
雌雄同体
阿姆斯壮
而我知道
赌神
别惹我
九号球
武装
时光机
我们 (时时刻刻)
在这一秒
生命有一种绝对
王子面
小时候
孙悟空
倔强
垃圾车
小护士
让我照顾你
约翰蓝侬
回来吧
错错错
晚安，地球人
超人
神的孩子都在跳舞
圣诞夜惊魂
垃圾车(朋友版)
前传
为爱而生
天使
我又初恋了
香水
摩托车日记
最重要的小事
快乐很伟大
忘词
宠上天
米老鼠
一千个世纪
胎音
突然好想你
生存以上 生活以下
你不是真正的快乐
爆肝
噢买尬
出头天
我心中尚未崩坏的地方
春天的呐喊
夜访吸血鬼
如烟
后青春期的诗
笑忘歌
有些事现在不做 一辈子都不会做了
我不愿让你一个人
星空
洗衣机
三个傻瓜
歪腰
干杯
仓颉
2012
第二人生
诺亚方舟
明日
OAOA (现在就是永远)
T1213121
末日
OAOA (丢掉名字性别)
如果我们不曾相遇
成名在望
好好 (想把你写成一首歌)
兄弟
人生有限公司
后来的我们
顽固
派对动物
最好的一天
少年他的奇幻漂流
终于结束的起点
任意门
转眼
知足
牙关
乱世浮生
恋爱ing
听不到
温柔 (还你自由版)
金多虾
麦来乱
OK 啦
垃圾车 (朋友版)
未来 (Sailing With Me)
咸鱼


# 词频与词性分析

In [32]:
def process_lyrics_with_jieba(text):
    # 1. 词性标注与分词
    # jieba.posseg 会同时返回词和词性
    words_with_pos = pseg.cut(text)

    
    # 2. 过滤无意义字符（标点、空格、单字符停用词）
    filtered_data = []
    for word, pos in words_with_pos:
        # 排除标点符号（x表示标点）及空白字符
        if pos != 'x' and len(word.strip()) > 0:
            filtered_data.append((word, pos))
    
    # 3. 统计词频
    word_counts = Counter([item[0] for item in filtered_data])
    
    # 4. 汇总信息 (词, 词性, 频数)
    # 我们以词为 Key，存储词性
    word_pos_map = {word: pos for word, pos in filtered_data}
    
    # 排序：按词频从高到低
    sorted_results = []
    for word, count in word_counts.most_common():
        sorted_results.append({
            "词": word,
            "词性": word_pos_map[word],
            "频数": count
        })
    
    return sorted_results

In [34]:
# 读取歌词文件
with open("output/mayday_lyric_260124.json", 'r') as f:
    lyric_data = json.load(f)

In [40]:
lyric_words_dict = {}
for i in lyric_data:
    if i:
        lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(i['歌词'])
lyric_words_dict

{'386925': [{'词': '我', '词性': 'r', '频数': 18},
  {'词': '好想', '词性': 'v', '频数': 18},
  {'词': '那么', '词性': 'r', '频数': 12},
  {'词': '多', '词性': 'm', '频数': 12},
  {'词': '的', '词性': 'uj', '频数': 12},
  {'词': '这个', '词性': 'r', '频数': 9},
  {'词': '飞', '词性': 'v', '频数': 9},
  {'词': '逃离', '词性': 'v', '频数': 8},
  {'词': '你', '词性': 'r', '频数': 7},
  {'词': '了', '词性': 'ul', '频数': 6},
  {'词': '是', '词性': 'v', '频数': 6},
  {'词': '世界', '词性': 'n', '频数': 5},
  {'词': '疯狂世界', '词性': 'nw', '频数': 4},
  {'词': '苦', '词性': 'a', '频数': 4},
  {'词': '累', '词性': 'v', '频数': 4},
  {'词': '莫名', '词性': 'v', '频数': 4},
  {'词': '疯狂', '词性': 'a', '频数': 4},
  {'词': '如果', '词性': 'c', '频数': 4},
  {'词': '发现', '词性': 'v', '频数': 4},
  {'词': '也别', '词性': 'd', '频数': 4},
  {'词': '将', '词性': 'd', '频数': 4},
  {'词': '挽回', '词性': 'v', '频数': 4},
  {'词': '泪水', '词性': 'n', '频数': 3},
  {'词': '后悔', '词性': 'v', '频数': 2},
  {'词': '多么', '词性': 'r', '频数': 2},
  {'词': '伤悲', '词性': 'a', '频数': 2},
  {'词': '了解', '词性': 'v', '频数': 2},
  {'词': '在', '词性': 'p', '频数': 2},
  {'词': '用力

In [49]:
rows = []
for song_id, word_list in lyric_words_dict.items():
    for item in word_list:
        # 创建新字典，保留原始数据并加入歌曲ID列
        new_row = {
            'song_id': song_id,
            '词': item['词'],
            '词性': item['词性'],
            '频数': item['频数']
        }
        rows.append(new_row)

# 3. 转换为 DataFrame
df_word = pd.DataFrame(rows)
df_word

,song_id,词,词性,频数
0,386925,我,r,18
1,386925,好想,v,18
2,386925,那么,r,12
3,386925,多,m,12
4,386925,的,uj,12
...,...,...,...,...
12473,386040,吹风,v,1
12474,386040,要,v,1
12475,386040,一天,m,1
12476,386040,做,v,1


In [51]:
# 合并
# 1. 确保 df_word 的 song_id 是字符串
df_word['song_id'] = df_word['song_id'].astype(str)

# 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
df_unique['song_id'] = df_unique['song_id'].astype(str).str.strip()

# 3. 执行合并
df_merged = df_word.merge(df_unique, on='song_id', how='left')

df_merged

,song_id,词,词性,频数,album_order,album_id,album_name,release_date,song_order,song_name,album_fixed,heat_index
0,386925,我,r,18,1,38315,第一张创作专辑,1999-07-07,1,疯狂世界,第一张创作专辑,17.0
1,386925,好想,v,18,1,38315,第一张创作专辑,1999-07-07,1,疯狂世界,第一张创作专辑,17.0
2,386925,那么,r,12,1,38315,第一张创作专辑,1999-07-07,1,疯狂世界,第一张创作专辑,17.0
3,386925,多,m,12,1,38315,第一张创作专辑,1999-07-07,1,疯狂世界,第一张创作专辑,17.0
4,386925,的,uj,12,1,38315,第一张创作专辑,1999-07-07,1,疯狂世界,第一张创作专辑,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...
12473,386040,吹风,v,1,10,38247,知足 最真杰作选,2005-08-26,30,咸鱼,知足 最真杰作选,118.0
12474,386040,要,v,1,10,38247,知足 最真杰作选,2005-08-26,30,咸鱼,知足 最真杰作选,118.0
12475,386040,一天,m,1,10,38247,知足 最真杰作选,2005-08-26,30,咸鱼,知足 最真杰作选,118.0
12476,386040,做,v,1,10,38247,知足 最真杰作选,2005-08-26,30,咸鱼,知足 最真杰作选,118.0


In [53]:
df_merged.to_csv('output/mayday_lyric_word.csv')

In [43]:
with open('lyric_temp.json', 'w', encoding='utf-8') as f:
    json.dump(lyric_words_dict, f, ensure_ascii=False, indent=4)

In [45]:
def analyze_lyrics_counts(lyric_words_dict):
    # 1. 加载数据
    # with open(file_path, 'r', encoding='utf-8') as f:
    #     data = json.load(f)
    data = lyric_words_dict
    
    # 2. 初始化统计
    # stats 的结构: {(词, 词性): [出现次数, 频数累计]}
    stats = defaultdict(lambda: [0, 0])
    total_frequency_sum = 0 # 记录所有频数之和
    
    # 3. 遍历统计
    for song_id, word_list in data.items():
        for item in word_list:
            word = item['词']
            pos = item['词性']
            freq = item['频数']
            
            # 唯一标识符
            key = (word, pos)
            
            # “次数”：每在列表中出现一次，计数 +1
            stats[key][0] += 1
            # “频数”：累加歌曲中的具体数值
            stats[key][1] += freq
            # “频数之和”：全局累加
            total_frequency_sum += freq

    # 4. 转换并排序
    final_list = []
    for (word, pos), (occurrence, total_freq) in stats.items():
        final_list.append({
            "词": word,
            "词性": pos,
            "出现次数": occurrence, # 在 JSON 列表里出现的次数
            "累计频数": total_freq   # 频数数值的加总
        })
    
    # 按“累计频数”降序排列
    final_list.sort(key=lambda x: x['累计频数'], reverse=True)
    
    return final_list, total_frequency_sum

In [46]:
analyze_lyrics_counts(lyric_words_dict)

([{'词': '的', '词性': 'uj', '出现次数': 122, '累计频数': 1630},
  {'词': '我', '词性': 'r', '出现次数': 117, '累计频数': 1603},
  {'词': '你', '词性': 'r', '出现次数': 96, '累计频数': 1089},
  {'词': '是', '词性': 'v', '出现次数': 100, '累计频数': 378},
  {'词': '在', '词性': 'p', '出现次数': 83, '累计频数': 335},
  {'词': '了', '词性': 'ul', '出现次数': 86, '累计频数': 320},
  {'词': '有', '词性': 'v', '出现次数': 81, '累计频数': 311},
  {'词': '啦', '词性': 'y', '出现次数': 15, '累计频数': 230},
  {'词': '不', '词性': 'd', '出现次数': 65, '累计频数': 216},
  {'词': '都', '词性': 'd', '出现次数': 59, '累计频数': 198},
  {'词': '和', '词性': 'c', '出现次数': 52, '累计频数': 186},
  {'词': '着', '词性': 'uz', '出现次数': 59, '累计频数': 184},
  {'词': '谁', '词性': 'r', '出现次数': 42, '累计频数': 183},
  {'词': '这', '词性': 'r', '出现次数': 59, '累计频数': 176},
  {'词': '就', '词性': 'd', '出现次数': 63, '累计频数': 172},
  {'词': '要', '词性': 'v', '出现次数': 59, '累计频数': 167},
  {'词': '爱', '词性': 'v', '出现次数': 54, '累计频数': 163},
  {'词': '那', '词性': 'r', '出现次数': 53, '累计频数': 148},
  {'词': '我们', '词性': 'r', '出现次数': 35, '累计频数': 148},
  {'词': '也', '词性': 'd', '出现次数': 55, '累计频

## 整体词频统计

# 歌曲热度

In [ ]:
songs = df_unique['song_name'].to_list()

['疯狂世界',
 '拥抱',
 '透露',
 '生活',
 '爱情的模样',
 '嘿！我要走了',
 '轧车',
 '志明与春娇',
 'HoSee',
 '黑白讲',
 'I Love You 无望',
 '风若吹',
 '为什么 (今日的爱情)',
 '终结孤单',
 '明白',
 '心中无别人',
 '有你的将来',
 '憨人',
 '叫我第一名',
 '雨眠',
 '罗密欧与茱丽叶',
 '温柔',
 '爱情万岁',
 '反而',
 '一颗苹果',
 '能不能不要说',
 '好不好',
 '相信',
 'Ok啦',
 '借问众神明',
 '永远的永远',
 '彩虹',
 '啾啾啾',
 '纯真',
 '候鸟',
 '人生海海',
 '轻功',
 '恒星的恒心',
 '雌雄同体',
 '阿姆斯壮',
 '而我知道',
 '赌神',
 '别惹我',
 '九号球',
 '武装',
 '时光机',
 '我们 (时时刻刻)',
 '在这一秒',
 '生命有一种绝对',
 '王子面',
 '小时候',
 '孙悟空',
 '倔强',
 '垃圾车',
 '小护士',
 '让我照顾你',
 '约翰蓝侬',
 '回来吧',
 '错错错',
 '晚安，地球人',
 '超人',
 '神的孩子都在跳舞',
 '圣诞夜惊魂',
 '垃圾车(朋友版)',
 '前传',
 '为爱而生',
 '天使',
 '我又初恋了',
 '香水',
 '摩托车日记',
 '最重要的小事',
 '快乐很伟大',
 '忘词',
 '宠上天',
 '米老鼠',
 '一千个世纪',
 '胎音',
 '突然好想你',
 '生存以上 生活以下',
 '你不是真正的快乐',
 '爆肝',
 '噢买尬',
 '出头天',
 '我心中尚未崩坏的地方',
 '春天的呐喊',
 '夜访吸血鬼',
 '如烟',
 '后青春期的诗',
 '笑忘歌',
 '有些事现在不做 一辈子都不会做了',
 '我不愿让你一个人',
 '星空',
 '洗衣机',
 '三个傻瓜',
 '歪腰',
 '干杯',
 '仓颉',
 2012,
 '第二人生',
 '诺亚方舟',
 '明日',
 'OAOA (现在就是永远)',
 'T1213121',
 '末日',
 'OAOA (丢掉名字性别)',
 '如果我们不曾相遇',
 '成名在望',
 '

In [14]:
len(songs)

130

In [15]:
songs_heat_index = [
    {"歌名": "突然好想你", "排名": 1},
    {"歌名": "倔强", "排名": 2},
    {"歌名": "温柔", "排名": 3},
    {"歌名": "后来的我们", "排名": 4},
    {"歌名": "知足", "排名": 5},
    {"歌名": "你不是真正的快乐", "排名": 6},
    {"歌名": "恋爱ing", "排名": 7},
    {"歌名": "我不愿让你一个人", "排名": 8},
    {"歌名": "干杯", "排名": 9},
    {"歌名": "志明与春娇", "排名": 10},
    {"歌名": "如烟", "排名": 11},
    {"歌名": "成名在望", "排名": 12},
    {"歌名": "派对动物", "排名": 13},
    {"歌名": "OAOA (现在就是永远)", "排名": 14},
    {"歌名": "仓颉", "排名": 15},
    {"歌名": "顽固", "排名": 16},
    {"歌名": "疯狂世界", "排名": 17},
    {"歌名": "诺亚方舟", "排名": 18},
    {"歌名": "人生海海", "排名": 19},
    {"歌名": "星空", "排名": 20},
    {"歌名": "天使", "排名": 21},
    {"歌名": "最重要的小事", "排名": 22},
    {"歌名": "憨人", "排名": 23},
    {"歌名": "孙悟空", "排名": 24},
    {"歌名": "一颗苹果", "排名": 25},
    {"歌名": "终结孤单", "排名": 26},
    {"歌名": "拥抱", "排名": 27},
    {"歌名": "恒星的恒心", "排名": 28},
    {"歌名": "时光机", "排名": 29},
    {"歌名": "我心中尚未崩坏的地方", "排名": 30},
    {"歌名": "让我照顾你", "排名": 31},
    {"歌名": "彩虹", "排名": 32},
    {"歌名": "爱情万岁", "排名": 33},
    {"歌名": "轧车", "排名": 34},
    {"歌名": "垃圾车", "排名": 35},
    {"歌名": "有些事现在不做 一辈子都不会做了", "排名": 36},
    {"歌名": "纯真", "排名": 37},
    {"歌名": "温柔 (还你自由版)", "排名": 38},
    {"歌名": "出头天", "排名": 39},
    {"歌名": "生命有一种绝对", "排名": 40},
    {"歌名": "风若吹", "排名": 41},
    {"歌名": "黑白讲", "排名": 42},
    {"歌名": "为什么 (今日的爱情)", "排名": 43},
    {"歌名": "雨眠", "排名": 44},
    {"歌名": "罗密欧与茱丽叶", "排名": 45},
    {"歌名": "好不好", "排名": 46},
    {"歌名": "借问众神明", "排名": 47},
    {"歌名": "永远的永远", "排名": 48},
    {"歌名": "嘿！我要走了", "排名": 49},
    {"歌名": "透露", "排名": 50},
    {"歌名": "生活", "排名": 51},
    {"歌名": "爱情的模样", "排名": 52},
    {"歌名": "I Love You 无望", "排名": 53},
    {"歌名": "HoSee", "排名": 54},
    {"歌名": "啾啾啾", "排名": 55},
    {"歌名": "雌雄同体", "排名": 56},
    {"歌名": "阿姆斯壮", "排名": 57},
    {"歌名": "而我知道", "排名": 58},
    {"歌名": "赌神", "排名": 59},
    {"歌名": "别惹我", "排名": 60},
    {"歌名": "九号球", "排名": 61},
    {"歌名": "武装", "排名": 62},
    {"歌名": "我们 (时时刻刻)", "排名": 63},
    {"歌名": "在这一秒", "排名": 64},
    {"歌名": "王子面", "排名": 65},
    {"歌名": "小时候", "排名": 66},
    {"歌名": "小护士", "排名": 67},
    {"歌名": "约翰蓝侬", "排名": 68},
    {"歌名": "回来吧", "排名": 69},
    {"歌名": "错错错", "排名": 70},
    {"歌名": "晚安，地球人", "排名": 71},
    {"歌名": "超人", "排名": 72},
    {"歌名": "神的孩子都在跳舞", "排名": 73},
    {"歌名": "圣诞夜惊魂", "排名": 74},
    {"歌名": "垃圾车(朋友版)", "排名": 75},
    {"歌名": "前传", "排名": 76},
    {"歌名": "为爱而生", "排名": 77},
    {"歌名": "我又初恋了", "排名": 78},
    {"歌名": "香水", "排名": 79},
    {"歌名": "摩托车日记", "排名": 80},
    {"歌名": "快乐很伟大", "排名": 81},
    {"歌名": "忘词", "排名": 82},
    {"歌名": "宠上天", "排名": 83},
    {"歌名": "米老鼠", "排名": 84},
    {"歌名": "一千个世纪", "排名": 85},
    {"歌名": "胎音", "排名": 86},
    {"歌名": "生存以上 生活以下", "排名": 87},
    {"歌名": "爆肝", "排名": 88},
    {"歌名": "噢买尬", "排名": 89},
    {"歌名": "春天的呐喊", "排名": 90},
    {"歌名": "夜访吸血鬼", "排名": 91},
    {"歌名": "后青春期的诗", "排名": 92},
    {"歌名": "笑忘歌", "排名": 93},
    {"歌名": "洗衣机", "排名": 94},
    {"歌名": "三个傻瓜", "排名": 95},
    {"歌名": "歪腰", "排名": 96},
    {"歌名": "2012", "排名": 97},
    {"歌名": "第二人生", "排名": 98},
    {"歌名": "明日", "排名": 99},
    {"歌名": "OAOA (丢掉名字性别)", "排名": 100},
    {"歌名": "如果我们不曾相遇", "排名": 101},
    {"歌名": "好好 (想把你写成一首歌)", "排名": 102},
    {"歌名": "兄弟", "排名": 103},
    {"歌名": "人生有限公司", "排名": 104},
    {"歌名": "最好的一天", "排名": 105},
    {"歌名": "少年他的奇幻漂流", "排名": 106},
    {"歌名": "终于结束的起点", "排名": 107},
    {"歌名": "任意门", "排名": 108},
    {"歌名": "转眼", "排名": 109},
    {"歌名": "牙关", "排名": 110},
    {"歌名": "乱世浮生", "排名": 111},
    {"歌名": "听不到", "排名": 112},
    {"歌名": "金多虾", "排名": 113},
    {"歌名": "麦来乱", "排名": 114},
    {"歌名": "OK 啦", "排名": 115},
    {"歌名": "垃圾车 (朋友版)", "排名": 116},
    {"歌名": "未来 (Sailing With Me)", "排名": 117},
    {"歌名": "咸鱼", "排名": 118},
    {"歌名": "明白", "排名": 119},
    {"歌名": "心中无别人", "排名": 120},
    {"歌名": "有你的将来", "排名": 121},
    {"歌名": "叫我第一名", "排名": 122},
    {"歌名": "能不能不要说", "排名": 123},
    {"歌名": "相信", "排名": 124},
    {"歌名": "Ok啦", "排名": 125},
    {"歌名": "候鸟", "排名": 126},
    {"歌名": "轻功", "排名": 127},
    {"歌名": "反而", "排名": 128}
]

In [17]:
# 为df_unique添加heat_index列
songs_heat_dict = {}
for i in songs_heat_index:
    songs_heat_dict[i['歌名']] = i['排名']
songs_heat_dict

{'突然好想你': 1,
 '倔强': 2,
 '温柔': 3,
 '后来的我们': 4,
 '知足': 5,
 '你不是真正的快乐': 6,
 '恋爱ing': 7,
 '我不愿让你一个人': 8,
 '干杯': 9,
 '志明与春娇': 10,
 '如烟': 11,
 '成名在望': 12,
 '派对动物': 13,
 'OAOA (现在就是永远)': 14,
 '仓颉': 15,
 '顽固': 16,
 '疯狂世界': 17,
 '诺亚方舟': 18,
 '人生海海': 19,
 '星空': 20,
 '天使': 21,
 '最重要的小事': 22,
 '憨人': 23,
 '孙悟空': 24,
 '一颗苹果': 25,
 '终结孤单': 26,
 '拥抱': 27,
 '恒星的恒心': 28,
 '时光机': 29,
 '我心中尚未崩坏的地方': 30,
 '让我照顾你': 31,
 '彩虹': 32,
 '爱情万岁': 33,
 '轧车': 34,
 '垃圾车': 35,
 '有些事现在不做 一辈子都不会做了': 36,
 '纯真': 37,
 '温柔 (还你自由版)': 38,
 '出头天': 39,
 '生命有一种绝对': 40,
 '风若吹': 41,
 '黑白讲': 42,
 '为什么 (今日的爱情)': 43,
 '雨眠': 44,
 '罗密欧与茱丽叶': 45,
 '好不好': 46,
 '借问众神明': 47,
 '永远的永远': 48,
 '嘿！我要走了': 49,
 '透露': 50,
 '生活': 51,
 '爱情的模样': 52,
 'I Love You 无望': 53,
 'HoSee': 54,
 '啾啾啾': 55,
 '雌雄同体': 56,
 '阿姆斯壮': 57,
 '而我知道': 58,
 '赌神': 59,
 '别惹我': 60,
 '九号球': 61,
 '武装': 62,
 '我们 (时时刻刻)': 63,
 '在这一秒': 64,
 '王子面': 65,
 '小时候': 66,
 '小护士': 67,
 '约翰蓝侬': 68,
 '回来吧': 69,
 '错错错': 70,
 '晚安，地球人': 71,
 '超人': 72,
 '神的孩子都在跳舞': 73,
 '圣诞夜惊魂': 74,
 '垃圾车(朋友版)': 

In [26]:
df_unique['heat_index'] = df_unique['song_name'].map(songs_heat_dict).fillna(0)

In [27]:
df_unique

,album_order,album_id,album_name,release_date,song_order,song_id,song_name,album_fixed,heat_index
0,1,38315,第一张创作专辑,1999-07-07,1,386925,疯狂世界,第一张创作专辑,17.0
1,1,38315,第一张创作专辑,1999-07-07,2,386927,拥抱,第一张创作专辑,27.0
2,1,38315,第一张创作专辑,1999-07-07,3,386929,透露,第一张创作专辑,50.0
3,1,38315,第一张创作专辑,1999-07-07,4,386930,生活,第一张创作专辑,51.0
4,1,38315,第一张创作专辑,1999-07-07,5,386931,爱情的模样,第一张创作专辑,52.0
...,...,...,...,...,...,...,...,...,...
149,10,38247,知足 最真杰作选,2005-08-26,20,386010,麦来乱,知足 最真杰作选,114.0
153,10,38247,知足 最真杰作选,2005-08-26,24,386025,OK 啦,知足 最真杰作选,115.0
154,10,38247,知足 最真杰作选,2005-08-26,25,386028,垃圾车 (朋友版),知足 最真杰作选,116.0
158,10,38247,知足 最真杰作选,2005-08-26,29,386038,未来 (Sailing With Me),知足 最真杰作选,117.0
